# Dunnhumby M4-C 단독 · 마지막 1주 · 10시드

- 학습 DAY 1~704 / test DAY 705~711 / seed 42~51 / 고정 300 epoch
- 완료된 같은 마지막 주 M1·M3·M5-C 30개 결과는 재사용합니다.
- M4-C만 최대 10개 학습합니다. 완료 시드는 건너뛰고 중단 시드는 저장 epoch에서 재개합니다.
- 개발구간(684~690)의 M4-C seed42·44와 원형/분리형 M4 결과는 재사용하지 않습니다.
- 준비 셀의 실행계획표를 확인한 뒤 학습 셀을 실행하세요. L4 GPU를 권장합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임(L4 권장)을 연결하세요.'
SOURCE_COMMIT = 'a26ec38b911e962164955dddeeb9a0b765a75ad6'
REPO = Path('/content/clv-m4c-final10-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', torch.cuda.get_device_name(0), '| code:', SOURCE_COMMIT)


In [ ]:
import clv_m4_user_centered_final10 as screen
cfg = screen.configure()
print('참조 결과:', cfg.reference_dir)
print('M4-C 저장:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('동일 마지막주 M1/M3/M5-C 완료 결과:', len(prepared['references']['absolute']), '개 — 재학습 없음')
plan = screen.execution_plan(prepared)
print(plan.to_string(index=False))
print('최대 추가 학습 epoch 합계:', int(plan.max_remaining_epochs.sum()))
print('추가학습 없는 시드:', int(plan.action.eq('reuse_completed').sum()))
print('준비 완료 — 다음 셀에서 M4-C의 빠진 학습만 시작합니다.')


In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


In [ ]:
from IPython.display import display
metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10',
           'price_purchase_amount_weighted_hit@20','vndcg@20',
           'price_purchase_amount_weighted_hit@50','vndcg@50']
view = result['summary'][result['summary'].metric.isin(metrics)]
display(view.pivot(index='metric', columns='model_id', values='mean').reindex(metrics))
display(view.pivot(index='metric', columns='model_id', values='std').reindex(metrics))
print('상호작용 = (M5-C−M3)−(M4-C−M1), 기술적 요인비교이며 유의성을 주장하지 않습니다.')
display(result['factorial'].groupby('metric').interaction.agg(['mean','std']).reindex(metrics))
print('전체 세그먼트·노출·불리한 지표는 CSV/JSON에 모두 저장됩니다.')


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
output = Path('/content/clv_m4_user_centered_final10_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
    archive.write(prepared['run_dir'] / 'protocol.json', 'protocol.json')
files.download(str(output))
